# Análisis de mercados: baldosas cerámicas (HS 690722) para Corona
Proyecto final de Data Mining and Business Intelligence. Este cuaderno reproduce todos los resultados del dashboard: limpia los datos, calcula los cinco indicadores, arma el ranking, prueba escenarios de pesos y guarda los CSV que usa el dashboard.

**Cómo usarlo:** sube a la misma carpeta los dos archivos Excel de entrada (`UNComtrade-trade-statistics-official-CAHS-20261121-1517.xlsx` y `anex-EXPORTACIONES-SerieDestino-sep2026.xlsx`) y ejecuta las celdas en orden.

**Fuentes y unidades:** UN Comtrade (importaciones reportadas por cada país, USD corrientes, peso en kg) y DANE (exportaciones totales de Colombia, millones de USD).

In [ ]:
import pandas as pd
import numpy as np
ARCH_COMTRADE = "UNComtrade-trade-statistics-official-CAHS-20261121-1517.xlsx"
ARCH_DANE = "anex-EXPORTACIONES-SerieDestino-sep2026.xlsx"
ANIOS = [2022, 2023, 2024]

## 1. Cargar y limpiar Comtrade

In [ ]:
com = pd.read_excel(ARCH_COMTRADE)
com = com[["refYear","reporterDesc","partnerCode","partnerDesc","primaryValue","netWgt"]]
com.columns = ["anio","pais","socio_cod","socio","valor_usd","peso_kg"]
total = com[com.socio_cod == 0].copy()   # total que importa cada pais
prov = com[com.socio_cod != 0].copy()    # detalle por proveedor
print("filas:", com.shape[0], "| totales:", total.shape[0], "| proveedores:", prov.shape[0])
print(sorted(com.pais.unique()), sorted(com.anio.unique()))

### Verificación: la suma de proveedores debe coincidir con el total de cada país y año

In [ ]:
chequeo = prov.groupby(["pais","anio"]).valor_usd.sum().to_frame("suma_proveedores")
chequeo["total_reportado"] = total.set_index(["pais","anio"]).valor_usd
chequeo["diferencia"] = chequeo.suma_proveedores - chequeo.total_reportado
print(chequeo.round(0))
print("Países sin peso (kg) reportado:", total[total.peso_kg.isna() | (total.peso_kg == 0)].pais.unique())

## 2. Cargar el total de exportaciones de Colombia (DANE)

In [ ]:
d = pd.read_excel(ARCH_DANE, header=None)
dane = pd.DataFrame({"anio": d.iloc[8,1:].tolist(), "exp_colombia_musd": d.iloc[9,1:].tolist()})
dane = dane[dane.anio.isin(ANIOS)].copy()
dane["anio"] = dane.anio.astype(int)
print(dane)

## 3. Indicadores por país
Tamaño (promedio anual en USD), crecimiento (CAGR 2022-2024), estabilidad (coeficiente de variación; menor es mejor), concentración de proveedores (HHI de 2024; menor es mejor) y participación de Colombia (valor que vende Colombia dividido entre el total importado, promedio de los 3 años).

In [ ]:
t = total.pivot_table(index="pais", columns="anio", values="valor_usd", aggfunc="sum")
col = prov[prov.socio == "Colombia"].pivot_table(index="pais", columns="anio", values="valor_usd", aggfunc="sum").reindex(t.index).fillna(0)
hhi = {}
for p, g in prov[prov.anio == 2024].groupby("pais"):
    s = g.groupby("socio").valor_usd.sum()
    hhi[p] = ((s / s.sum() * 100) ** 2).sum()
base = pd.DataFrame({
    "Pais_Importador": t.index,
    "Tamaño_Promedio_USD": t.mean(axis=1).values,
    "Crecimiento_CAGR": ((t[2024] / t[2022]) ** 0.5 - 1).values,
    "Estabilidad_CV": (t.std(axis=1, ddof=0) / t.mean(axis=1)).values,
    "Concentracion_HHI_2024": [hhi[p] for p in t.index],
    "Participacion_Colombia_Prom": (col / t).mean(axis=1).values})
base.round(4)

### Verificación manual (crecimiento de Guatemala)

In [ ]:
g24, g22 = t.loc["Guatemala", 2024], t.loc["Guatemala", 2022]
print("CAGR Guatemala a mano:", round((g24 / g22) ** 0.5 - 1, 4))
print("CAGR Guatemala en la tabla:", round(base.set_index("Pais_Importador").loc["Guatemala","Crecimiento_CAGR"], 4))

## 4. Ranking (pesos base 25/25/15/15/20) y sensibilidad

In [ ]:
def minmax(s): return (s - s.min()) / (s.max() - s.min()) * 100
n = pd.DataFrame({
    "Pais_Importador": base.Pais_Importador,
    "Tamaño": minmax(base.Tamaño_Promedio_USD),
    "Crecimiento": minmax(base.Crecimiento_CAGR),
    "Estabilidad": 100 - minmax(base.Estabilidad_CV),          # menor CV es mejor
    "Concentracion": 100 - minmax(base.Concentracion_HHI_2024), # menor HHI es mejor
    "Participacion_Col": minmax(base.Participacion_Colombia_Prom)})
PESOS = {  # tamaño, crecimiento, estabilidad, concentración, participación Colombia
    "Base": [.25, .25, .15, .15, .20],
    "Prudente": [.20, .10, .35, .25, .10],
    "Agresivo": [.25, .45, .05, .05, .20],
    "Foco Colombia": [.15, .15, .10, .10, .50]}
ranking = base.merge(n, on="Pais_Importador")
for nombre, w in PESOS.items():
    ranking["Puntaje_" + nombre] = n.iloc[:, 1:].values @ np.array(w)
ranking = ranking.sort_values("Puntaje_Base", ascending=False)
cols = ["Pais_Importador"] + [c for c in ranking.columns if c.startswith("Puntaje")]
print(ranking[cols].round(1).to_string(index=False))
print("\nPrimer lugar por escenario:")
for nombre in PESOS: print(nombre, "->", ranking.sort_values("Puntaje_" + nombre, ascending=False).Pais_Importador.iloc[0])

## 5. Bases anuales para los filtros del dashboard

In [ ]:
filas = []
tt = total.set_index(["pais","anio"])
for (pa, a), g in prov.groupby(["pais","anio"]):
    s = g.groupby("socio").valor_usd.sum()
    v = tt.loc[(pa, a)]
    c = s.get("Colombia", 0)
    filas.append(dict(pais=pa, anio=a, importaciones_usd=v.valor_usd, peso_kg=v.peso_kg,
                      valor_colombia_usd=c, participacion_colombia=c / v.valor_usd,
                      hhi=((s / s.sum() * 100) ** 2).sum(), n_proveedores=len(s)))
anual = pd.DataFrame(filas)
anual.round(3).head(6)

## 6. Guardar los archivos que usa el dashboard

In [ ]:
base.to_csv("base_procesada_indicadores_corregida.csv", index=False, encoding="utf-8-sig")
ranking.to_csv("ranking_final_mercados_corregido.csv", index=False, encoding="utf-8-sig")
anual.to_csv("base_anual_paises.csv", index=False, encoding="utf-8-sig")
prov[["anio","pais","socio","valor_usd","peso_kg"]].to_csv("importaciones_por_proveedor.csv", index=False, encoding="utf-8-sig")
dane.to_csv("exportaciones_colombia_dane.csv", index=False)
print("Archivos guardados.")

## 7. Cifras de la presentación: precio por kg, competencia en Costa Rica y flete
Estas celdas producen los datos de la diapositiva de riesgos: valor unitario de importación (USD/kg), cambio de Costa Rica en dólares frente a kilos, participación y precio de Brasil frente a Colombia, y el peso relativo del flete. El 0,03 USD/kg de flete es un supuesto de ejemplo, no una cotización.

In [ ]:
# Precio por kg (valor unitario de importación) por país y año
t_kg = total.assign(peso_kg=total.peso_kg.replace(0, np.nan))
t_kg["usd_kg"] = t_kg.valor_usd / t_kg.peso_kg
print("Valor unitario de importación (USD/kg):")
print(t_kg.pivot_table(index="pais", columns="anio", values="usd_kg").round(3))

# Costa Rica: ¿cayó el volumen o el precio?
cr = t_kg[t_kg.pais == "Costa Rica"].set_index("anio")
print(f"\nCosta Rica 2022-2024: cambio en USD {cr.valor_usd[2024] / cr.valor_usd[2022] - 1:.1%}"
      f" | cambio en kg {cr.peso_kg[2024] / cr.peso_kg[2022] - 1:.1%}")

# Costa Rica: participación y precio de Brasil frente a Colombia
bc = prov[(prov.pais == "Costa Rica") & prov.socio.isin(["Brazil", "Colombia"])].copy()
bc["participacion_%"] = bc.valor_usd / bc.anio.map(cr.valor_usd) * 100
bc["usd_kg"] = bc.valor_usd / bc.peso_kg
print("\nCosta Rica, Brasil vs. Colombia:")
print(bc.pivot_table(index="anio", columns="socio", values=["participacion_%", "usd_kg"]).round(3))

# Peso del flete: cuánto representa 0,03 USD/kg sobre el precio promedio de Costa Rica en 2024
print(f"\nCada 0,03 USD/kg de flete equivale a {0.03 / cr.usd_kg[2024]:.1%} del valor de importación de Costa Rica en 2024")
